# 📈 Cuaderno 02 — Integración numérica

**Métodos Numéricos · Unidad 5: Derivadas, integrales y EDO**

Temas **5.2 Integración numérica** (trapecio, Simpson 1/3 y 3/8) y
**5.3 Integración con intervalos desiguales** (capítulo 21 del libro).

Aprendes a usar `np.trapezoid` y `scipy.integrate.simpson` (y su equivalente
MATLAB `trapz`), entendiendo qué calculan, qué reciben, qué devuelven y
cuándo conviene cada regla.

> Camino: **teoría → laboratorio → entradas/salidas → preguntas → caso →
> envío**.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
import os, sys

CUADERNO = 'u5_02_integracion'
TAREA_ID = 'U5_02_Integracion_Numerica'
REPO_NOMBRE = 'U5 derivadas integrales y EDO'

def _raiz_local():
    b = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(b, 'lib', 'robomat.py')):
            return b
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

if ES_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = os.path.join('/content/drive/MyDrive', 'CursoMN', REPO_NOMBRE)
else:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError('Abre el cuaderno dentro de ' + REPO_NOMBRE)

sys.path.insert(0, os.path.join(REPO, 'lib'))
print('Modo       :', 'Colab' if ES_COLAB else 'local')
print('Repositorio:', REPO)

In [ ]:
#@title 📦 Librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

exec(open(os.path.join(REPO, 'lib', 'robomat.py'), encoding='utf-8').read(), globals())

## 1. Teoría: integrar con reglas de cuadratura

La integral definida $I = \int_a^b f(x)\,dx$ es el **área bajo la curva**.
Cuando no hay antiderivada cerrada o solo tenemos datos, la aproximamos con
**reglas de cuadratura** que reemplazan $f$ por un polinomio interpolante.

### Regla del trapecio (polinomio de grado 1)

Un segmento:  $$I \cong (b-a)\,\frac{f(a)+f(b)}{2} \tag{21.3}$$

Múltiples segmentos ($n$ intervalos, $h=\frac{b-a}{n}$):

$$I = \frac{h}{2}\left[f(x_0) + 2\sum_{i=1}^{n-1} f(x_i) + f(x_n)\right] \tag{21.9}$$

### Regla de Simpson 1/3 (parábola, grado 2)

$$I \cong \frac{h}{3}\left[f(x_0)+4f(x_1)+f(x_2)\right] \tag{21.14}$$

Requiere **número par de segmentos**. Es más exacta que el trapecio con el
mismo $h$ (error $O(h^4)$ vs $O(h^2)$).

### Regla de Simpson 3/8 (cúbica, grado 3)

$$I \cong \frac{3h}{8}\left[f(x_0)+3f(x_1)+3f(x_2)+f(x_3)\right] \tag{21.20}$$

Requiere **múltiplo de 3 segmentos**; se combina con 1/3 para $n$ impar.

### Intervalos desiguales

Con datos irregularmente espaciados, cada segmento usa su propio $h_i$:

$$I = h_1\frac{f(x_0)+f(x_1)}{2} + h_2\frac{f(x_1)+f(x_2)}{2} + \dots + h_n\frac{f(x_{n-1})+f(x_n)}{2} \tag{21.22}$$

> 📖 **Figuras del libro:** `fig_21_4.pdf` (trapecio), `fig_21_10`
> (Simpson 1/3 y 3/8), `fig_21_12.pdf` (combinar 1/3 + 3/8),
> `fig_21_14` (datos irregulares).

## 2. Laboratorio: integrar con NumPy/SciPy

Integra $f(x) = 0.2 + 25x - 200x^2 + 675x^3 - 900x^4 + 400x^5$ en $[0, 0.8]$
(ejemplo 21.1–21.6 del libro). La integral **exacta es 1.640533**.

In [ ]:
def f(x):
    return 0.2 + 25*x - 200*x**2 + 675*x**3 - 900*x**4 + 400*x**5

a, b = 0.0, 0.8
exacta = 1.640533

# Trapecio con n=2 segmentos (ejemplo 21.2)
x2 = np.linspace(a, b, 3)
I_trap = np.trapezoid(f(x2), x2)

# Simpson 1/3 con n=4 segmentos (ejemplo 21.5)
x4 = np.linspace(a, b, 5)
I_s13 = integrate.simpson(f(x4), x4)

# Simpson 3/8 con n=3 segmentos (ejemplo 21.6)
x3 = np.linspace(a, b, 4)
I_s38 = integrate.simpson(f(x3), x3)  # simpson acepta cualquier n

print('Método          | Integral  | error')
print(f'Trapecio (n=2)  | {I_trap:8.6f} | {abs(I_trap-exacta):.2e}')
print(f'Simpson 1/3 (n=4)| {I_s13:8.6f} | {abs(I_s13-exacta):.2e}')
print(f'Simpson 3/8 (n=3)| {I_s38:8.6f} | {abs(I_s38-exacta):.2e}')

In [ ]:
xx = np.linspace(a, b, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(xx, f(xx), 'k-', label='f(x)')
plt.fill_between(x2, 0, f(x2), alpha=0.25, label='trapecio (n=2)')
plt.fill_between(x4, 0, f(x4), alpha=0.35, label='Simpson 1/3 (n=4)')
plt.xlabel('x'); plt.ylabel('f(x)')
plt.legend(); plt.grid(alpha=0.3); plt.title('Reglas de cuadratura')
plt.show()

## 3. Entradas, salidas y cuándo usar cada una

### `np.trapezoid(y, x=None, dx=1.0)`

- **Entrada:** `y` (valores del integrando) y `x` (abscisas, **opcional**; si
  se omite usa `dx` uniforme). **Acepta `x` irregular**.
- **Salida:** un `float` con el área (trapecio).
- **Cuándo:** datos tabulados, o integración rápida; es la base de intervalos
  desiguales (21.22).

### `scipy.integrate.simpson(y, x=None, dx=1.0, even='avg')`

- **Entrada:** `y`, `x` (opcional) y `even` (qué hacer si hay segmentos
  impares: `'avg'` combina 1/3 y 3/8, `'first'`/`'last'` descarta un
  extremo).
- **Salida:** un `float` (Simpson 1/3, o combinación 1/3+3/8).
- **Cuándo:** más precisión con el mismo número de puntos que el trapecio;
  requiere función suave (el error depende de $f^{(4)}$).

> 🐍 **MATLAB:** `trapz(x, y)` ≈ `np.trapezoid`; `integral(f, a, b)` es
> adaptativa (equivale a `scipy.integrate.quad`).

**Regla práctica:** trapecio para datos o rapidez; Simpson cuando tienes una
función suave evaluable en puntos equiespaciados y quieres error menor.

## 4. Preguntas teóricas

**P1.** La regla de Simpson 1/3 interpola $f$ con un polinomio de grado…

1. 1 (recta)
2. 2 (parábola)
3. 3 (cúbica)
4. 4 (cuártica)

**P2.** Para aplicar Simpson 1/3 de aplicación múltiple, el número de
segmentos debe ser…

1. cualquier entero positivo
2. impar
3. par
4. múltiplo de 3

In [ ]:
RESPUESTA_P1 = 2   # Simpson 1/3 usa una parábola
RESPUESTA_P2 = 3   # número par de segmentos

## 5. Caso de ingeniería: datos con intervalos desiguales

Un medidor de flujo registra el caudal $f(x)$ de un río en instantes de
tiempo **no uniformes**. El volumen descargado es $V = \int f\,dx$. Con la
tabla 21.3 del libro:

| $x$ | 0.00 | 0.12 | 0.22 | 0.32 | 0.36 | 0.40 | 0.44 | 0.54 | 0.64 | 0.70 | 0.80 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| $f(x)$ | 0.200 | 1.310 | 1.305 | 1.743 | 2.075 | 2.456 | 2.843 | 3.507 | 3.182 | 2.363 | 0.232 |

Usa `np.trapezoid` (que acepta `x` irregular) y compara con el valor exacto
1.640533.

In [ ]:
x = np.array([0.0, 0.12, 0.22, 0.32, 0.36, 0.40, 0.44, 0.54, 0.64, 0.70, 0.80])
y = np.array([0.200000, 1.309729, 1.305241, 1.743393, 2.074903, 2.456000,
              2.842985, 3.507297, 3.181929, 2.363000, 0.232000])

V = np.trapezoid(y, x)          # trapecio con intervalos desiguales (21.22)
exacta = 1.640533

print(f'Volumen (trapecio desigual) = {V:.6f}')
print(f'Error relativo = {abs(V - exacta)/exacta*100:.2f} %')

RESPUESTA_CASO = float(V)

## 6. Revisar, enviar y ver calificación

Las respuestas (`RESPUESTA_P1`, `RESPUESTA_P2`, `RESPUESTA_CASO`) se envían
al **Apps Script** (se conecta en la Fase 4).

In [ ]:
#@title 📤 Enviar respuestas y ver calificación
alumno_id = 'TU_NUMERO_CONTROL'
respuestas = {'P1': RESPUESTA_P1, 'P2': RESPUESTA_P2, 'CASO': RESPUESTA_CASO}
print('Pendiente de conectar con Apps Script:', respuestas)